In [ ]:
import pandas as pd
import numpy as np

In [ ]:
sheet_id = '1PSws1P-N6tiT1yLCmq-nSiE8YcKV0SXvsQLbtK70Zfs'

sheet_20260212_S1 = '20260212_S1'
url_20260212_S1 = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_20260212_S1}"

sheet_20260212_S2 = '20260212_S2'
url_20260212_S2 = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_20260212_S2}"

In [ ]:
def process_stock_dataframe(
    df,
    time_col='Time',
    code_col='Code',
    price_col='Price',
    action_col='Action',
    lot_col='Lot',
    market_col='Market',
    sort_by='Price'
):
    """
    Memproses dataframe transaksi saham:
    - Ambil data terakhir per Code
    - Split Price & Change
    - Hitung BuyVolume, SellVolume, Volume, Net
    - Return dataframe final siap analisis
    """

    df = df.copy()
    
    df[time_col] = pd.to_datetime(df[time_col])
    
    df_code = (
        df.sort_values(time_col)
          .drop_duplicates(subset=code_col, keep='last')
          .reset_index(drop=True)
    )
    
    df_code[[price_col, 'Change']] = (
        df_code[price_col]
        .astype(str)
        .str.extract(r'(\d+)\s*\(([+-]?\d+\.\d+%)\)')
    )

    df_code[price_col] = pd.to_numeric(df_code[price_col], errors='coerce')

    df_code['Change'] = (
        df_code['Change']
        .str.replace('%', '', regex=False)
        .astype(float)
    )
    
    df[lot_col] = (
        df[lot_col]
        .astype(str)
        .str.replace(',', '', regex=False)
    )

    df[lot_col] = pd.to_numeric(df[lot_col], errors='coerce')
    
    df['BuyLot'] = np.where(df[action_col] == 'Buy', df[lot_col], 0)
    df['SellLot'] = np.where(df[action_col] == 'Sell', df[lot_col], 0)

    summary = (
        df.groupby(code_col)
          .agg(
              BuyVolume=('BuyLot', 'sum'),
              SellVolume=('SellLot', 'sum')
          )
          .reset_index()
    )

    summary['Volume'] = summary['BuyVolume'] + summary['SellVolume']
    summary['Net'] = summary['BuyVolume'] - summary['SellVolume']
    
    df_final = df_code.merge(summary, on=code_col, how='left')
    
    if sort_by in df_final.columns:
        df_final = df_final.sort_values(sort_by).reset_index(drop=True)

    return df_final


In [ ]:
df_20260212_S1 = pd.read_csv(url_20260212_S1)

df_20260212_S1_final = process_stock_dataframe(df_20260212_S1)

df_20260212_S1_final


# SESI - 2 2026-02-12

In [ ]:
df_20260212_S2 = pd.read_csv(url_20260212_S2)

df_20260212_S2_final = process_stock_dataframe(df_20260212_S2)

df_20260212_S2_final
